In [ ]:
# Execute the SQL file to register functions (run this in Databricks)
# Uncomment and run in a Databricks notebook with spark available

"""
from pathlib import Path
import re

sql_file = Path("/Workspace/path/to/genie_space_table_functions.sql")
sql_content = sql_file.read_text()

# Split by CREATE OR REPLACE FUNCTION and execute each
statements = re.split(r'(?=CREATE OR REPLACE FUNCTION)', sql_content)

for stmt in statements:
    stmt = stmt.strip()
    if stmt.startswith('CREATE OR REPLACE FUNCTION'):
        # Find the end of the function (ends with $$;)
        if '$$;' in stmt:
            func_stmt = stmt[:stmt.index('$$;') + 3]
            print(f"Registering function...")
            spark.sql(func_stmt)
            print("Done!")
"""

print("Copy the above code to a Databricks notebook cell and uncomment to register functions")

### Register the Functions in Databricks

Execute the SQL file to create all functions. You can either:
1. Copy the SQL from `genie_space_table_functions.sql` to a Databricks SQL editor
2. Use `spark.sql()` to execute each CREATE FUNCTION statement
3. Use the Databricks CLI: `databricks sql execute --file genie_space_table_functions.sql`

In [ ]:
# Edge case test queries

edge_case_queries = """
-- ============================================================
-- ERROR TEST 1: Invalid table identifier format
-- Expected: ERROR message about invalid format
-- ============================================================
SELECT main_catalog.dev.add_table_to_worldbank_genie_space(
    'invalid_format'
) AS result;

-- ============================================================
-- ERROR TEST 2: Missing schema in identifier
-- Expected: ERROR message about invalid format
-- ============================================================
SELECT main_catalog.dev.add_table_to_worldbank_genie_space(
    'catalog.table_only'
) AS result;

-- ============================================================
-- IDEMPOTENCY TEST: Add same table twice
-- Expected: First call returns SUCCESS, second returns INFO (already exists)
-- ============================================================
SELECT main_catalog.dev.add_table_to_worldbank_genie_space(
    'main_catalog.worldbank.idempotency_test'
) AS first_add;

SELECT main_catalog.dev.add_table_to_worldbank_genie_space(
    'main_catalog.worldbank.idempotency_test'
) AS second_add;

-- ============================================================
-- CLEANUP: Remove the test table
-- ============================================================
SELECT main_catalog.dev.remove_table_from_genie_space(
    '<space_id>',
    'main_catalog.worldbank.idempotency_test'
) AS cleanup;
"""

print(edge_case_queries)

### Test 4: Edge Cases and Error Handling

In [ ]:
# SQL Test Queries (run these in Databricks SQL or via spark.sql)

test_queries = """
-- ============================================================
-- TEST 1: Add a table using the convenience function
-- ============================================================
SELECT main_catalog.dev.add_table_to_worldbank_genie_space(
    'main_catalog.worldbank.test_table'
) AS result;

-- ============================================================
-- TEST 2: Add a table by space_id (replace <space_id>)
-- ============================================================
SELECT main_catalog.dev.add_table_to_genie_space(
    '<space_id>',
    'main_catalog.worldbank.another_test_table'
) AS result;

-- ============================================================
-- TEST 3: Add table with column configurations
-- ============================================================
SELECT main_catalog.dev.add_table_with_columns_to_genie_space(
    '<space_id>',
    'main_catalog.worldbank.indicator_data',
    'indicator_id:true:false,indicator_name:true:true,value:false:false'
) AS result;

-- ============================================================
-- TEST 4: List all tables in the Genie Space
-- ============================================================
SELECT main_catalog.dev.list_tables_in_genie_space('<space_id>') AS tables;

-- ============================================================
-- TEST 5: Remove a table from the Genie Space
-- ============================================================
SELECT main_catalog.dev.remove_table_from_genie_space(
    '<space_id>',
    'main_catalog.worldbank.test_table'
) AS result;

-- ============================================================
-- TEST 6: Verify table was removed
-- ============================================================
SELECT main_catalog.dev.list_tables_in_genie_space('<space_id>') AS tables;
"""

print(test_queries)
print(f"\n-- Replace <space_id> with: {worldbank_space_id}")

### Test 3: SQL Queries to Test the Functions

Run these SQL queries in Databricks after registering the functions.

In [ ]:
# %sql
# -- Run this in a SQL cell or via spark.sql()
# SELECT main_catalog.dev.list_tables_in_genie_space('<space_id>')

# Python equivalent:
import json

if worldbank_space_id:
    space = client.genie.get_space(worldbank_space_id, include_serialized_space=True)
    if space.serialized_space:
        data = json.loads(space.serialized_space)
        tables = data.get("data_sources", {}).get("tables", [])
        print(f"Current tables in '{WORLDBANK_SPACE_TITLE}':")
        for t in tables:
            print(f"  - {t.get('identifier')}")

### Test 2: List Current Tables in the Genie Space

In [ ]:
# Find the Worldbank Table Finder Genie Space
# Import the space definition to get the title
from genie_space_definitions import WORLDBANK_TABLE_FINDER
from databricks.sdk import WorkspaceClient

client = WorkspaceClient()

# Use the title from the definitions file (single source of truth)
WORLDBANK_SPACE_TITLE = WORLDBANK_TABLE_FINDER.title
worldbank_space_id = None

print(f"Looking for Genie Space: '{WORLDBANK_SPACE_TITLE}'")
print("-" * 50)

spaces = client.genie.list_spaces()
for space in spaces.spaces or []:
    if space.title == WORLDBANK_SPACE_TITLE:
        worldbank_space_id = space.space_id
        print(f"Found Genie Space: {space.title}")
        print(f"Space ID: {space.space_id}")
        break

if not worldbank_space_id:
    print(f"Genie Space '{WORLDBANK_SPACE_TITLE}' not found")
    print("You may need to create it first using WORLDBANK_TABLE_FINDER.create(client)")

### Test 1: Find the Worldbank Genie Space ID

In [ ]:
# Read and display the SQL function definitions
from pathlib import Path

sql_file = Path("genie_space_table_functions.sql")
if sql_file.exists():
    print(sql_file.read_text()[:2000])
    print("\n... (truncated)")
else:
    print(f"SQL file not found at {sql_file.absolute()}")
    print("Make sure you're in the correct directory or provide the full path")

### Step 1: Register the SQL Functions

Run these SQL statements in Databricks to create the functions. 
You can execute them via `spark.sql()` or in a SQL notebook.

# Databricks Genie Space Migration

## Mermaid Diagram

```mermaid
flowchart TB
    subgraph SOURCE["SOURCE WORKSPACE (Dev)"]
        direction TB
        GS1[("Genie Space<br/>─────────<br/>space_id<br/>title<br/>warehouse_id<br/>serialized_space")]
        SDK1["WorkspaceClient<br/>Python SDK"]
        GS1 -->|"genie.get_space()<br/>include_serialized_space=True"| SDK1
    end

    subgraph EXPORT["EXPORT PIPELINE"]
        direction TB
        SERIAL["Serialize to JSON<br/>─────────<br/>json.dumps()"]
        VALIDATE1["Schema Validation<br/>─────────<br/>version check<br/>data_sources check"]
        SDK1 --> SERIAL
        SERIAL --> VALIDATE1
    end

    subgraph STORAGE["VERSION CONTROL"]
        direction TB
        JSON[("genie_space.json<br/>─────────<br/>space_id<br/>title<br/>source_workspace<br/>serialized_space")]
        GIT["Git Repository<br/>─────────<br/>commit<br/>push"]
        VALIDATE1 --> JSON
        JSON --> GIT
    end

    subgraph CICD["CI/CD PIPELINE"]
        direction TB
        TRIGGER["Pipeline Trigger<br/>─────────<br/>PR merge / manual"]
        CHECKOUT["Checkout Config<br/>─────────<br/>git pull"]
        VALIDATE2["Config Validation<br/>─────────<br/>JSON schema<br/>warehouse mapping"]
        GIT --> TRIGGER
        TRIGGER --> CHECKOUT
        CHECKOUT --> VALIDATE2
    end

    subgraph IMPORT["IMPORT PIPELINE"]
        direction TB
        PARSE["Parse JSON<br/>─────────<br/>json.loads()"]
        SDK2["WorkspaceClient<br/>Python SDK"]
        VALIDATE2 --> PARSE
        PARSE --> SDK2
    end

    subgraph TARGET["TARGET WORKSPACE (Prod)"]
        direction TB
        CHECK["Check Existing<br/>─────────<br/>genie.list_spaces()"]
        CREATE["Create Space<br/>─────────<br/>genie.create_space()"]
        UPDATE["Update Space<br/>─────────<br/>genie.update_space()"]
        GS2[("Genie Space<br/>─────────<br/>space_id<br/>title<br/>warehouse_id<br/>serialized_space")]
        SDK2 --> CHECK
        CHECK -->|"not exists"| CREATE
        CHECK -->|"exists"| UPDATE
        CREATE --> GS2
        UPDATE --> GS2
    end

    style SOURCE fill:#FF3621,color:#fff
    style TARGET fill:#FF3621,color:#fff
    style STORAGE fill:#00AA55,color:#fff
    style CICD fill:#0066CC,color:#fff
    style EXPORT fill:#6B7280,color:#fff
    style IMPORT fill:#6B7280,color:#fff
```

## Nano Banana Pro Diagram Prompt

```
Create a modern CI/CD pipeline diagram for Databricks Genie Space migration with these components flowing left-to-right:

STAGE 1 - SOURCE (Orange #FF3621):
- Box: "SOURCE WORKSPACE (Dev)" containing:
  - Cylinder: "Genie Space" with fields: space_id, title, warehouse_id, serialized_space
  - Box: "WorkspaceClient SDK" with "genie.get_space()" label

STAGE 2 - EXPORT (Gray #6B7280):
- Box: "Serialize to JSON" with "json.dumps()" 
- Box: "Schema Validation" with checkmarks for version, data_sources

STAGE 3 - VERSION CONTROL (Green #00AA55):
- Document icon: "genie_space.json" showing JSON structure
- Box: "Git Repository" with git branch icon, "commit & push" label

STAGE 4 - CI/CD PIPELINE (Blue #0066CC):
- Box: "Pipeline Trigger" with play icon, "PR merge / manual dispatch"
- Box: "Checkout Config" with download icon
- Box: "Config Validation" with shield icon, "JSON schema, warehouse mapping"

STAGE 5 - IMPORT (Gray #6B7280):
- Box: "Parse JSON" with "json.loads()"
- Box: "WorkspaceClient SDK" 

STAGE 6 - TARGET (Orange #FF3621):
- Box: "TARGET WORKSPACE (Prod)" containing:
  - Decision diamond: "Space Exists?" with two paths
  - Box: "create_space()" for new
  - Box: "update_space()" for existing
  - Cylinder: "Genie Space" (deployed)

Flow arrows connecting each stage sequentially. Use rounded rectangles, modern flat design, subtle shadows. Add small icons: database for workspaces, document for JSON, git branch for repo, rocket for deploy. No legend needed - the flow is self-explanatory.
```

---

This notebook demonstrates how to use the Databricks SDK to migrate Genie Spaces between workspaces using a CI/CD approach.

## Install Latest Databricks SDK

## Setup and Authentication

The WorkspaceClient will automatically use authentication from:
1. Environment variables (`DATABRICKS_HOST`, `DATABRICKS_TOKEN`)
2. Databricks CLI config (`~/.databrickscfg`)
3. Azure CLI (if on Azure Databricks)

In [14]:
from databricks.sdk import WorkspaceClient


GENIE_SPACE_NAME = 'WHO Data Explorer Genie Space'
SOURCE_WORKSPACE = "https://dbc-930eaa5c-35a0.cloud.databricks.com"
TARGET_WORKSPACE = "https://dbc-c7690892-938f.cloud.databricks.com"

# Initialize the client (uses default authentication)
source_ws = WorkspaceClient(host=SOURCE_WORKSPACE)

## List Available Genie Spaces

First, let's see what Genie spaces are available.

In [15]:
# List all available Genie spaces
spaces_response = source_ws.genie.list_spaces()
for space in spaces_response.spaces:
    if space.title == GENIE_SPACE_NAME:
        SPACE_ID = space.space_id
        print(f"Space ID: {space.space_id}")
        print(f"Title: {space.title}")
        print(f"Description: {space.description}")
        print("-" * 50)
        

Space ID: 01f0d8eb90771a3ea44202866b79cb57
Title: WHO Data Explorer Genie Space
Description: None
--------------------------------------------------


## Get Genie Space Details

Use `get_space` to retrieve details of a specific Genie space.

In [16]:
# Get space details
space = source_ws.genie.get_space(space_id=SPACE_ID)
print(f"Space ID: {space.space_id}")
print(f"Title: {space.title}")
print(f"Description: {space.description}")
print(f"Warehouse ID: {space.warehouse_id}")

Space ID: 01f0d8eb90771a3ea44202866b79cb57
Title: WHO Data Explorer Genie Space
Description: None
Warehouse ID: 785fcb451a18a6d9


## Get Space with Serialized Space Data

Set `include_serialized_space=True` to get the full serialized space payload (requires CAN EDIT permission).

In [17]:
# Get space with serialized data (requires CAN EDIT permission)
space_full = source_ws.genie.get_space(
    space_id=SPACE_ID,
    include_serialized_space=True
)

print(f"Space ID: {space_full.space_id}")
print(f"Title: {space_full.title}")
print(f"Serialized Space: {space_full.serialized_space[:200] if space_full.serialized_space else 'N/A'}...")

Space ID: 01f0d8eb90771a3ea44202866b79cb57
Title: WHO Data Explorer Genie Space
Serialized Space: {
  "version": 1,
  "data_sources": {
    "tables": [
      {
        "identifier": "main_catalog.dev.worldbank_indicators",
        "column_configs": [
          {
            "column_name": "aggrega...


## Export Serialized Space to JSON File

Save the serialized space configuration to a JSON file for version control, backup, or transfer.

In [18]:
import json
from pathlib import Path

# Define the output file path
json_file_path = Path("genie_space.json")

# Parse the serialized space (it's already a JSON string) and save with formatting
serialized_data = json.loads(space_full.serialized_space)

# Create export payload with metadata
export_payload = {
    "space_id": space_full.space_id,
    "title": space_full.title,
    "description": space_full.description,
    "source_workspace": SOURCE_WORKSPACE,
    "serialized_space": serialized_data
}

# Write to JSON file
with open(json_file_path, "w") as f:
    json.dump(export_payload, f, indent=2)

print(f"Exported Genie space to: {json_file_path.absolute()}")
print(f"File size: {json_file_path.stat().st_size} bytes")

Exported Genie space to: /Users/casper/Documents/Vergence/repos/vector_search_index/genie_space.json
File size: 6201 bytes


## Read Serialized Space from JSON File

Load the Genie space configuration from the JSON file for import into target workspace.

In [19]:
# Read the exported JSON file
with open(json_file_path, "r") as f:
    imported_payload = json.load(f)

print(f"Loaded Genie space from: {json_file_path}")
print(f"Space Title: {imported_payload['title']}")
print(f"Source Workspace: {imported_payload['source_workspace']}")
print(f"Number of tables: {len(imported_payload['serialized_space']['data_sources']['tables'])}")

# Convert back to JSON string for API calls
imported_serialized_space = json.dumps(imported_payload['serialized_space'])

Loaded Genie space from: genie_space.json
Space Title: WHO Data Explorer Genie Space
Source Workspace: https://dbc-930eaa5c-35a0.cloud.databricks.com
Number of tables: 2


## Import Genie Space to Target Workspace

Create or update the Genie space in the target workspace using the imported JSON configuration.

In [20]:
target_ws = WorkspaceClient(host=TARGET_WORKSPACE)
warehouse_id = list(target_ws.warehouses.list())[0].id

# Check if a space with the same name already exists
existing_space_id = None
target_spaces = target_ws.genie.list_spaces()
for s in target_spaces.spaces:
    if s.title == imported_payload['title']:
        existing_space_id = s.space_id
        break

if existing_space_id:
    # Update existing space using imported JSON
    result = target_ws.genie.update_space(
        space_id=existing_space_id,
        title=imported_payload['title'],
        warehouse_id=warehouse_id,
        serialized_space=imported_serialized_space
    )
    print(f"Updated existing Genie space: {existing_space_id}")
else:
    # Create new space using imported JSON
    result = target_ws.genie.create_space(
        title=imported_payload['title'],
        warehouse_id=warehouse_id,
        serialized_space=imported_serialized_space
    )
    print(f"Created new Genie space: {result.space_id}")

result

Updated existing Genie space: 01f0d8f0127e1abbb1351d684d9d1321


GenieSpace(space_id='01f0d8f0127e1abbb1351d684d9d1321', title='WHO Data Explorer Genie Space', description=None, serialized_space='{\n  "version": 1,\n  "data_sources": {\n    "tables": [\n      {\n        "identifier": "main_catalog.dev.worldbank_indicators",\n        "column_configs": [\n          {\n            "column_name": "aggregation_method",\n            "get_example_values": true,\n            "build_value_dictionary": true\n          },\n          {\n            "column_name": "embedding_text",\n            "get_example_values": true,\n            "build_value_dictionary": true\n          },\n          {\n            "column_name": "indicator_id",\n            "get_example_values": true\n          },\n          {\n            "column_name": "indicator_name",\n            "get_example_values": true,\n            "build_value_dictionary": true\n          },\n          {\n            "column_name": "license_type",\n            "get_example_values": true,\n            "build_val

In [12]:
print(space_full.serialized_space)


{
  "version": 1,
  "data_sources": {
    "tables": [
      {
        "identifier": "main_catalog.dev.worldbank_indicators",
        "column_configs": [
          {
            "column_name": "aggregation_method",
            "get_example_values": true,
            "build_value_dictionary": true
          },
          {
            "column_name": "embedding_text",
            "get_example_values": true,
            "build_value_dictionary": true
          },
          {
            "column_name": "indicator_id",
            "get_example_values": true
          },
          {
            "column_name": "indicator_name",
            "get_example_values": true,
            "build_value_dictionary": true
          },
          {
            "column_name": "license_type",
            "get_example_values": true,
            "build_value_dictionary": true
          },
          {
            "column_name": "long_definition",
            "get_example_values": true,
            "build_value_dic

---

## Test SQL Functions for Genie Space Table Management

The following cells test the SQL UDF functions that manage tables in Genie Spaces.
These functions are defined in `genie_space_table_functions.sql`.